In [11]:
import pandas as pd
import numpy as np
import re
from s3_utils import read_s3_csv, get_s3_client,upload_to_s3

In [12]:
s3 = get_s3_client()
BUCKET = "projet-accidents-jedha"

# 1. On liste TOUS les fichiers du dossier bronze
response = s3.list_objects_v2(Bucket=BUCKET, Prefix='bronze/')

# 2. On crée une liste vide pour stocker nos DataFrames
dfs_to_merge = []

# 3. La boucle magique intelligente
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    # On filtre pour ne garder que la catégorie "caract"
    if "usagers" in file_key.lower():
        
        # --- LOGIQUE D'EXTRACTION DE L'ANNÉE ---
        # On cherche 4 chiffres dans le nom (ex: 2024)
        match = re.search(r'(\d{4})', file_key)
        
        if match:
            annee = int(match.group(1))
            
            # On définit le séparateur selon l'année
            if annee >= 2024:
                sep = ','
            else:
                sep = ';'
            
            print(f"🔎 Trouvé : {file_key} | Année: {annee} | Séparateur: '{sep}'")
            
            # On lit le fichier avec le bon séparateur
            df_temp = read_s3_csv(file_key, separator=sep)
            dfs_to_merge.append(df_temp)

# 4. On fusionne tout d'un coup
if dfs_to_merge:
    usagers_df = pd.concat(dfs_to_merge, ignore_index=True)
    print(f"✅ Terminé ! {len(dfs_to_merge)} fichiers fusionnés.")
else:
    print("⚠️ Aucun fichier trouvé, vérifie le mot-clé ou le dossier.")

🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - usagers 2021.csv | Année: 2021 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - usagers 2022.csv | Année: 2022 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - usagers 2023.csv | Année: 2023 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - usagers 2024.csv | Année: 2024 | Séparateur: ','
✅ Terminé ! 4 fichiers fusionnés.


In [13]:
# ── Chargement et merge des 4 fichiers CSV ──────────────────────
# On charge chaque année séparément car le séparateur de 2024 est différent
# 2021, 2022, 2023 → séparateur point-virgule (;)
# 2024 → séparateur virgule (,)
# On ajoute une colonne 'annee' pour pouvoir filtrer par année dans les analyses
"""# Puis on empile les 4 tables avec pd.concat

import pandas as pd
import numpy as np

df_2021 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2021.csv', sep=';')
df_2022 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2022.csv', sep=';')
df_2023 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2023.csv', sep=';')
df_2024 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2024.csv', sep=',')

df_2021['annee'] = 2021
df_2022['annee'] = 2022
df_2023['annee'] = 2023
df_2024['annee'] = 2024

usagers_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)
print(f"✅ Merge OK — {usagers_df.shape[0]:,} lignes x {usagers_df.shape[1]} colonnes")
usagers_df.head()"""

'# Puis on empile les 4 tables avec pd.concat\n\nimport pandas as pd\nimport numpy as np\n\ndf_2021 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - usagers 2021.csv\', sep=\';\')\ndf_2022 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - usagers 2022.csv\', sep=\';\')\ndf_2023 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - usagers 2023.csv\', sep=\';\')\ndf_2024 = pd.read_csv(\'Dataset/Dataset - Sécurité Routière.xlsx - usagers 2024.csv\', sep=\',\')\n\ndf_2021[\'annee\'] = 2021\ndf_2022[\'annee\'] = 2022\ndf_2023[\'annee\'] = 2023\ndf_2024[\'annee\'] = 2024\n\nusagers_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)\nprint(f"✅ Merge OK — {usagers_df.shape[0]:,} lignes x {usagers_df.shape[1]} colonnes")\nusagers_df.head()'

In [14]:
# ── Remplacement des -1 par NaN ─────────────────────────────────
# Dans le fichier BAAC de l'ONISR, la valeur -1 signifie "non renseigné"
# Ce n'est pas une vraie valeur numérique, on la remplace par NaN
cols_sentinel = ['place', 'grav', 'sexe', 'trajet',
                 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']

usagers_df[cols_sentinel] = usagers_df[cols_sentinel].replace(-1, np.nan)
print("✅ Valeurs -1 remplacées par NaN")

✅ Valeurs -1 remplacées par NaN


In [15]:
# ── Correction des années de naissance aberrantes ───────────────
usagers_df.loc[usagers_df['an_nais'] > 2010, 'an_nais'] = np.nan
print(f"✅ an_nais corrigé")
print(f"Min : {usagers_df['an_nais'].min()}  |  Max : {usagers_df['an_nais'].max()}")
print(f"NaN : {usagers_df['an_nais'].isna().sum()}")

✅ an_nais corrigé
Min : 1912.0  |  Max : 2010.0
NaN : 30811


In [16]:
# ── Suppression des colonnes inutiles ───────────────────────────
usagers_df = usagers_df.drop(columns=['place', 'etatp'])
print(f"✅ Colonnes supprimées")
print(f"Colonnes restantes : {list(usagers_df.columns)}")

✅ Colonnes supprimées
Colonnes restantes : ['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp']


In [17]:
# ── Vérification finale avant export ────────────────────────────
print(f"Shape final : {usagers_df.shape}")

print("\n--- NaN par colonne ---")
nan_df = pd.DataFrame({
    'NaN count': usagers_df.isna().sum(),
    'NaN %': (usagers_df.isna().mean() * 100).round(1)
})
print(nan_df[nan_df['NaN count'] > 0])

print("\n--- Vérif aucun -1 restant ---")
for col in usagers_df.select_dtypes(include='number').columns:
    n = (usagers_df[col] == -1).sum()
    if n > 0:
        print(f"  ⚠️  {col} : {n} valeurs -1 restantes")
print("✅ Aucun -1 restant")

print("\n--- Doublons ---")
print(f"Doublons : {usagers_df.duplicated(['Num_Acc','id_usager']).sum()}")

Shape final : (506886, 14)

--- NaN par colonne ---
         NaN count  NaN %
grav           419    0.1
sexe         10632    2.1
an_nais      30811    6.1
trajet       11269    2.2
secu1         9865    1.9
secu2       223921   44.2
secu3       489711   96.6
locp        240813   47.5

--- Vérif aucun -1 restant ---
✅ Aucun -1 restant

--- Doublons ---
Doublons : 0


In [18]:
"""# ── Export du fichier nettoyé ───────────────────────────────────
usagers_df.to_csv('Dataset/usagers_2021_2024_clean.csv', index=False, sep=';')
print(f"✅ Exporté — {len(usagers_df):,} lignes x {usagers_df.shape[1]} colonnes")"""

'# ── Export du fichier nettoyé ───────────────────────────────────\nusagers_df.to_csv(\'Dataset/usagers_2021_2024_clean.csv\', index=False, sep=\';\')\nprint(f"✅ Exporté — {len(usagers_df):,} lignes x {usagers_df.shape[1]} colonnes")'

In [19]:
# On utilise ta fonction d'upload
# 'lieux_cleaned.csv_test' est le nom que le fichier aura sur S3
upload_to_s3(usagers_df, "usagers_cleaned_test.csv", folder="silver")

✅ Mission accomplie : silver/usagers_cleaned_test.csv est sur S3 !
